# Image Captioning with CNN + Transformer

Here trains an image captioning model on the Flickr8k dataset using a frozen EfficientNetB0 visual encoder coupled to a Transformer encoder/decoder.



In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if not gpus:
    print('\nWARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/image_captioning'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Will save outputs to:', SAVE_DIR)

## 3. Set up the Kaggle API



In [ ]:
from google.colab import files
uploaded = files.upload()  # select kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
!pip install -q kaggle

## 4. Download and extract the Flickr8k dataset

In [ ]:
!kaggle datasets download -d adityajn105/flickr8k
!unzip -qq flickr8k.zip
!ls

## 5. Convert the data to the expected format



In [ ]:
import pandas as pd

if os.path.isdir('Images') and not os.path.isdir('Flicker8k_Dataset'):
    os.rename('Images', 'Flicker8k_Dataset')

df = pd.read_csv('captions.txt')
print('First few rows of captions.txt:')
print(df.head())
print(f'Total caption rows: {len(df)}')

with open('Flickr8k.token.txt', 'w') as f:
    for img_name, group in df.groupby('image'):
        for i, caption in enumerate(group['caption']):
            f.write(f'{img_name}#{i}\t{caption}\n')

print('\nFirst few lines of Flickr8k.token.txt:')
!head -3 Flickr8k.token.txt
print('\nNumber of images in Flicker8k_Dataset:')
!ls Flicker8k_Dataset | wc -l

## 6. Imports and configuration

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import efficientnet
from tensorflow.keras.layers import TextVectorization

SEED = 111
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMAGES_PATH = 'Flicker8k_Dataset'
IMAGE_SIZE = (299, 299)
VOCAB_SIZE = 10000
SEQ_LENGTH = 25
EMBED_DIM = 512
FF_DIM = 512
BATCH_SIZE = 64
EPOCHS = 30
AUTOTUNE = tf.data.AUTOTUNE

print('Config loaded.')

## 7. Caption loading and split

In [ ]:
def load_captions_data(filename):
    with open(filename) as f:
        caption_data = f.readlines()

    caption_mapping = {}
    text_data = []
    images_to_skip = set()

    for line in caption_data:
        line = line.rstrip('\n')
        img_name, caption = line.split('\t')
        img_name = img_name.split('#')[0]
        img_name = os.path.join(IMAGES_PATH, img_name.strip())

        tokens = caption.strip().split()
        if len(tokens) < 5 or len(tokens) > SEQ_LENGTH:
            images_to_skip.add(img_name)
            continue

        if img_name.endswith('jpg') and img_name not in images_to_skip:
            caption = '<start> ' + caption.strip() + ' <end>'
            text_data.append(caption)
            caption_mapping.setdefault(img_name, []).append(caption)

    for img_name in images_to_skip:
        caption_mapping.pop(img_name, None)

    return caption_mapping, text_data


def train_val_split(caption_data, train_size=0.8, shuffle=True):
    all_images = list(caption_data.keys())
    if shuffle:
        np.random.shuffle(all_images)
    n_train = int(len(all_images) * train_size)
    training_data = {k: caption_data[k] for k in all_images[:n_train]}
    validation_data = {k: caption_data[k] for k in all_images[n_train:]}
    return training_data, validation_data


captions_mapping, text_data = load_captions_data('Flickr8k.token.txt')
train_data, valid_data = train_val_split(captions_mapping)
print(f'Training samples:   {len(train_data)}')
print(f'Validation samples: {len(valid_data)}')
print(f'Total captions:     {len(text_data)}')

## 8. Text vectorization

In [ ]:
strip_chars = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
strip_chars = strip_chars.replace('<', '').replace('>', '')

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(strip_chars), '')

vectorization = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=SEQ_LENGTH,
    standardize=custom_standardization,
)
vectorization.adapt(text_data)

print(f'Vocabulary size: {len(vectorization.get_vocabulary())}')
print(f'First 20 tokens: {vectorization.get_vocabulary()[:20]}')

## 9. Image pipeline and dataset

In [ ]:
image_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomContrast(0.3),
])

def decode_and_resize(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

def process_input(img_path, captions):
    return decode_and_resize(img_path), vectorization(captions)

def make_dataset(images, captions):
    ds = tf.data.Dataset.from_tensor_slices((images, captions))
    ds = ds.shuffle(BATCH_SIZE * 8)
    ds = ds.map(process_input, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_dataset = make_dataset(list(train_data.keys()), list(train_data.values()))
valid_dataset = make_dataset(list(valid_data.keys()), list(valid_data.values()))
print(f'Train batches: {len(train_dataset)}')
print(f'Valid batches: {len(valid_dataset)}')

## 10. Model definition

In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs) * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)


def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)
    return keras.models.Model(base_model.input, base_model_out)


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_proj = layers.Dense(embed_dim, activation='relu')

    def call(self, inputs, training=False, mask=None):
        x = self.dense_proj(inputs)
        x = self.layernorm_1(x)
        attn_out = self.attention(query=x, value=x, key=x, attention_mask=None, training=training)
        return self.layernorm_2(x + attn_out)


class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim, activation='relu')
        self.ffn_layer_2 = layers.Dense(embed_dim)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.embedding = PositionalEmbedding(
            embed_dim=EMBED_DIM, sequence_length=SEQ_LENGTH, vocab_size=VOCAB_SIZE)
        self.out = layers.Dense(VOCAB_SIZE, activation='softmax')
        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training=False, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)

        if mask is not None:
            padding_mask = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(combined_mask, causal_mask)
        else:
            padding_mask = None
            combined_mask = causal_mask

        attn_1 = self.attention_1(query=inputs, value=inputs, key=inputs,
                                  attention_mask=combined_mask, training=training)
        out_1 = self.layernorm_1(inputs + attn_1)

        attn_2 = self.attention_2(query=out_1, value=encoder_outputs, key=encoder_outputs,
                                  attention_mask=padding_mask, training=training)
        out_2 = self.layernorm_2(out_1 + attn_2)

        ffn = self.ffn_layer_1(out_2)
        ffn = self.dropout_1(ffn, training=training)
        ffn = self.ffn_layer_2(ffn)
        ffn = self.layernorm_3(ffn + out_2)
        ffn = self.dropout_2(ffn, training=training)
        return self.out(ffn)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype='int32')
        mask = tf.reshape(mask, (1, sequence_length, sequence_length))
        mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)


class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5, image_aug=None):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name='loss')
        self.acc_tracker = keras.metrics.Mean(name='accuracy')
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug

    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def _compute_caption_loss_and_acc(self, img_embed, batch_seq, training=True):
        encoder_out = self.encoder(img_embed, training=training)
        batch_seq_inp = batch_seq[:, :-1]
        batch_seq_true = batch_seq[:, 1:]
        mask = tf.math.not_equal(batch_seq_true, 0)
        batch_seq_pred = self.decoder(batch_seq_inp, encoder_out, training=training, mask=mask)
        loss = self.calculate_loss(batch_seq_true, batch_seq_pred, mask)
        acc = self.calculate_accuracy(batch_seq_true, batch_seq_pred, mask)
        return loss, acc

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0.0
        batch_acc = 0.0

        if self.image_aug is not None:
            batch_img = self.image_aug(batch_img)

        img_embed = self.cnn_model(batch_img)

        with tf.GradientTape() as tape:
            for i in range(self.num_captions_per_image):
                loss, acc = self._compute_caption_loss_and_acc(
                    img_embed, batch_seq[:, i, :], training=True)
                batch_loss += loss
                batch_acc += acc
            avg_loss = batch_loss / float(self.num_captions_per_image)

        train_vars = self.encoder.trainable_variables + self.decoder.trainable_variables
        grads = tape.gradient(avg_loss, train_vars)
        self.optimizer.apply_gradients(zip(grads, train_vars))

        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {'loss': self.loss_tracker.result(), 'acc': self.acc_tracker.result()}

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0.0
        batch_acc = 0.0
        img_embed = self.cnn_model(batch_img)
        for i in range(self.num_captions_per_image):
            loss, acc = self._compute_caption_loss_and_acc(
                img_embed, batch_seq[:, i, :], training=False)
            batch_loss += loss
            batch_acc += acc
        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {'loss': self.loss_tracker.result(), 'acc': self.acc_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]


class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_lr = self.post_warmup_learning_rate * (global_step / warmup_steps)
        return tf.cond(global_step < warmup_steps,
                       lambda: warmup_lr,
                       lambda: self.post_warmup_learning_rate)

print('Model classes defined.')

## 11. Build and compile the model

In [ ]:
cnn_model = get_cnn_model()
encoder = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=1)
decoder = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=2)
caption_model = ImageCaptioningModel(
    cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=image_augmentation)

cross_entropy = keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')
early_stopping = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

num_train_steps = len(train_dataset) * EPOCHS
num_warmup_steps = num_train_steps // 15
lr_schedule = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps)

caption_model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=cross_entropy,
)
print('Model compiled.')

## 12. Smoke test

In [ ]:
_ = caption_model.fit(
    train_dataset,
    epochs=1,
    validation_data=valid_dataset,
)

## 13. Full training (29 more epochs)



In [ ]:
import os
import numpy as np
import pickle

SAVE_DIR = '/content/drive/MyDrive/image_captioning'
os.makedirs(SAVE_DIR, exist_ok=True)

class SaveEachEpoch(keras.callbacks.Callback):
    """Save encoder + decoder weights as NumPy arrays after each epoch.
    Works regardless of whether the components are Layers or Models."""

    def on_epoch_end(self, epoch, logs=None):
        try:
            # Get all weight arrays from encoder and decoder
            encoder_weights = [w.numpy() for w in self.model.encoder.weights]
            decoder_weights = [w.numpy() for w in self.model.decoder.weights]

            np.savez(
                os.path.join(SAVE_DIR, 'encoder_weights.npz'),
                *encoder_weights
            )
            np.savez(
                os.path.join(SAVE_DIR, 'decoder_weights.npz'),
                *decoder_weights
            )
            print(f'  [Saved checkpoint after epoch {epoch+1}]')
        except Exception as e:
            print(f'  [Save failed: {e}]')

# Save vocabulary once
with open(os.path.join(SAVE_DIR, 'vocabulary.pkl'), 'wb') as f:
    pickle.dump(vectorization.get_vocabulary(), f)
print('Saved vocabulary.')

history = caption_model.fit(
    train_dataset,
    epochs=30,
    validation_data=valid_dataset,
    callbacks=[early_stopping, SaveEachEpoch()],
)

## 14. Save model weights to Google Drive



In [ ]:
import os
import numpy as np
import pickle

SAVE_DIR = '/content/drive/MyDrive/image_captioning'
os.makedirs(SAVE_DIR, exist_ok=True)

# Extract weights as NumPy arrays (works on any Keras object, Layer or Model)
encoder_weights = [w.numpy() for w in caption_model.encoder.weights]
decoder_weights = [w.numpy() for w in caption_model.decoder.weights]

# Save as .npz (numpy archive)
encoder_path = os.path.join(SAVE_DIR, 'encoder_weights.npz')
decoder_path = os.path.join(SAVE_DIR, 'decoder_weights.npz')
np.savez(encoder_path, *encoder_weights)
np.savez(decoder_path, *decoder_weights)

print('Saved encoder weights to:', encoder_path)
print('Saved decoder weights to:', decoder_path)

# Save the vocabulary
vocab_path = os.path.join(SAVE_DIR, 'vocabulary.pkl')
with open(vocab_path, 'wb') as f:
    pickle.dump(vectorization.get_vocabulary(), f)
print('Saved vocabulary to:', vocab_path)

# Verify all files exist
print('\nFiles in SAVE_DIR:')
for f in sorted(os.listdir(SAVE_DIR)):
    full = os.path.join(SAVE_DIR, f)
    size_mb = os.path.getsize(full) / 1024 / 1024
    print(f'  {f}: {size_mb:.2f} MB')

## 15. Generate captions for sample images



In [ ]:
vocab = vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocab)), vocab))
max_decoded_sentence_length = SEQ_LENGTH - 1
valid_images = list(valid_data.keys())

def generate_caption(save_path=None):
    sample_img_path = np.random.choice(valid_images)
    sample_img = decode_and_resize(sample_img_path)

    # Fix display: normalize to [0, 1] regardless of source range
    img_for_display = sample_img.numpy()
    if img_for_display.max() > 1.0:
        img_for_display = img_for_display / 255.0
    img_for_display = np.clip(img_for_display, 0.0, 1.0)

    plt.figure(figsize=(8, 8))
    plt.imshow(img_for_display)
    plt.axis('off')

    img = tf.expand_dims(sample_img, 0)
    img = caption_model.cnn_model(img)
    encoded_img = caption_model.encoder(img, training=False)

    decoded_caption = '<start> '
    for i in range(max_decoded_sentence_length):
        tokenized = vectorization([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized, 0)
        predictions = caption_model.decoder(
            tokenized, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_lookup[sampled_token_index]
        if sampled_token == '<end>':
            break
        decoded_caption += ' ' + sampled_token

    caption = decoded_caption.replace('<start> ', '').replace(' <end>', '').strip()
    plt.title(f'Predicted: {caption}', fontsize=14)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150, facecolor='white')
    plt.show()
    print(f'Image: {os.path.basename(sample_img_path)}')
    print(f'Caption: {caption}\n')
    return caption

# Re-generate the 5 sample predictions with proper image display
for i in range(5):
    save_path = os.path.join(SAVE_DIR, f'sample_prediction_{i+1}.png')
    generate_caption(save_path=save_path)

print(f'Sample predictions saved to {SAVE_DIR}')